## Imports

In [26]:
from pynq.lib.video import *
import numpy as np
import threading
import time
from pynq.overlays.base import BaseOverlay
from datetime import datetime
base = BaseOverlay("base.bit")
btns = base.btns_gpio

## Functions and set up

In [27]:
%%microblaze base.PMODB

#include "gpio.h"
#include "pyprintf.h"

//Function to turn on/off a selected pin of PMODB
int write_gpio(unsigned int pin, unsigned int val){
    if (val > 1){
        pyprintf("pin value must be 0 or 1");
    }
    gpio pin_out = gpio_open(pin);
    gpio_set_direction(pin_out, GPIO_OUT);
    gpio_write(pin_out, val);
    return 0;
}

//Function to read the value of a selected pin of PMODB
unsigned int read_gpio(unsigned int pin){
    gpio pin_in = gpio_open(pin);
    gpio_set_direction(pin_in, GPIO_IN);
    return gpio_read(pin_in);
}

void clear_gpios(){
    for(int i = 0; i < 8; ++i){
        write_gpio(i,0);
    }
}

In [28]:
clear_gpios()

In [29]:
def write_gpio_pwm(pin, brightness, duration=0.5, freq=200):
    """
    pin: RBG -> pins 2, 1, 3 respectively
    brightness: 0–100
    duration: time LED stays on (s)
    freq: PWM frequency (Hz)
    """
    if brightness < 0: brightness = 0
    if brightness > 100: brightness = 100

    period = 1.0 / freq
    on_time = period * (brightness / 100.0)
    off_time = period - on_time

    end_time = time.time() + duration

    while time.time() < end_time:
        write_gpio(pin, 1)
        time.sleep(on_time)
        write_gpio(pin, 0)
        time.sleep(off_time)
    write_gpio(pin, 0)

In [30]:
# set led functions, incorporate this to sample the state every 0.1s
def set_led_blue(time=0.1):
    write_gpio_pwm(1,50,duration=time)

def set_led_red():
    write_gpio_pwm(2,50,duration=time)

def set_led_green():
    write_gpio_pwm(3,50,duration=time)

In [34]:
def get_color(region):
    if region.size == 0:
        return np.array([0, 0, 0])

    # Reshape to list of pixels
    pixels = region.reshape(-1, 3)

    # Remove very dark pixels (noise/background)
    pixels = pixels[np.mean(pixels, axis=1) > 20]

    if len(pixels) == 0:
        return np.array([0, 0, 0])

    # Compute mean color (fast + stable)
    avg_color = np.mean(pixels, axis=0)

    return int(avg_color.astype)

## Threads

In [31]:
# thread set up
#   shared resource + lock + stop event
shared = {
    "face_count": 0,
    "frame": None,
    "captured_frame": None
}

lock = threading.Lock()
stop_event = threading.Event()

In [38]:
# ml thread
face_cascade = cv2.CascadeClassifier(
    "/home/xilinx/jupyter_notebooks/base/video/data/haarcascade_frontalface_default.xml"
)

def face_thread():
    cap = cv2.VideoCapture(0)

    if not cap.isOpened():
        print("Camera failed")
        return

    # Request high resolution from camera
    cap.set(cv2.CAP_PROP_FRAME_WIDTH, 1280)
    cap.set(cv2.CAP_PROP_FRAME_HEIGHT, 720)

    time.sleep(1)  # allow camera to stabilize

    while not stop_event.is_set():
        ret, frame = cap.read()
        if not ret:
            print("Frame read failed")
            time.sleep(0.5)
            continue

        # 🔹 Downscale ONLY for detection
        small = cv2.resize(frame, (320, 240))
        gray = cv2.cvtColor(small, cv2.COLOR_BGR2GRAY)

        faces = face_cascade.detectMultiScale(gray, 1.2, 3)

        with lock:
            shared["face_count"] = len(faces)
            shared["frame"] = small   # lightweight frame for UI

            # Save ONE high-res frame when face detected
            if len(faces) > 0:
                shared["captured_frame"] = frame.copy()

        time.sleep(0.05)   # 20 FPS max (stable)

    cap.release()

In [33]:
# RGB LED thread
def led_thread():
    single_face_start = None

    while not stop_event.is_set():
        with lock:
            count = shared["face_count"]

        if count == 0:
            set_led_red()
            single_face_start = None
        elif count > 1:
            set_led_blue()
            single_face_start = None
        elif count == 1:
            set_led_green()
            if single_face_start is None:
                single_face_start = time.time()
            elif time.time() - single_face_start >= 3:
                print("Stable single face detected")

                # blink 3 times
                # on the third blink, take pic
                for _ in range(3):
                    clear_gpios()
                    set_led_green(0.5)

                with lock:
                    shared["captured_frame"] = shared["frame"]

                stop_event.set()

## MAIN

In [39]:
if __name__ == "__main__":
    # Start threads
    t_face = threading.Thread(target=face_thread)
    t_led = threading.Thread(target=led_thread)

    t_face.start()
    t_led.start()

    try:
        # Main loop: just monitor stop_event and sleep
        while not stop_event.is_set():
            time.sleep(0.05)  # low CPU usage
    except KeyboardInterrupt:
        stop_event.set()

    # Wait for threads to finish
    t_face.join()
    t_led.join()

    # ==========================
    # Process captured frame colors
    # ==========================
    if shared["captured_frame"] is not None:
        print("Processing captured frame for final colors...")
        frame = shared["captured_frame"]
        gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
        faces = face_cascade.detectMultiScale(gray, 1.3, 5)

        if len(faces) > 0:
            x, y, w, h = faces[0]

            # Cheek region for skin
            cheek_region = frame[y + h//3 : y + h//2, x + w//4 : x + 3*w//4]

            # Hair region above forehead
            hair_region = frame[max(0, y - h//3):y, x:x+w]

            # Eye region (left half)
            eye_region = frame[y:y+h//2, x:x+w//2]

            print("Skin Color:", get_color(cheek_region))
            print("Hair Color:", get_color(hair_region))
            print("Eye Color:", get_color(eye_region))

    print("Done.")

[ WARN:9] global ./modules/videoio/src/cap_gstreamer.cpp (616) isPipelinePlaying OpenCV | GStreamer warning: GStreamer: pipeline have not been created


Done.
